# Velvet & Yeast Patisserie, Flavor Trend Dashboard

**Limited Edition Production Calendar Analysis**

This notebook explores consumers flavor preference trends to help the product manager indetify which flavors are growing and which are fading.

- **Dataset note:** No single clean public dataset exists for ingredient-level seatch trends overime. I generated a realistic synthethic dataset based on known food trend reporting(e.g. whole foods treand reports, Google Trends directional patterns). The shapes and trajectories are modeled on real-world flavor momentum, traditional flavors declining slowly, botanical/global ingredients rising sharply post-2018 

### 1. Imports

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')
print('Imports done!')

Imports done!


### 2. Generating the Dataset

 Each flavor gets a 'popularity index' score from 0-100 for each year from 2014 to 2024. The trend shapes are based on real food industry reporting.

In [4]:
np.random.seed(42)
years = list(range(2014, 2025))

# Each flavor: (category, starting_score, yearly_Change, noise_level)
flavor_config = {
    # Traditional — slowly fading
    'Vanilla':    ('Traditional', 88, -1.8, 2.0),
    'Chocolate':  ('Traditional', 85, -1.2, 2.5),
    'Strawberry': ('Traditional', 75, -1.5, 2.0),
    'Lemon':      ('Traditional', 70, -0.8, 1.5),
    # Floral — rising
    'Lavender':   ('Floral', 30, +3.5, 2.0),
    'Rose':       ('Floral', 28, +3.0, 2.5),
    'Elderflower':('Floral', 18, +4.2, 1.8),
    # Citrus/Global — rising fast
    'Yuzu':       ('Citrus/Global', 12, +5.8, 2.0),
    'Ube':        ('Citrus/Global', 10, +6.5, 2.2),
    'Passion Fruit':('Citrus/Global', 22, +4.0, 2.0),
    # Spice/Nutty — mixed
    'Cardamom':   ('Spice/Nutty', 20, +4.8, 1.5),
    'Matcha':     ('Spice/Nutty', 25, +5.2, 2.0),
    'Pistachio':  ('Spice/Nutty', 35, +3.8, 1.8),
    'Cinnamon':   ('Spice/Nutty', 60, -0.5, 2.0),
}

rows = []
for flavor, (category, start, change, noise) in flavor_config.items():
    for i, year in enumerate(years):
        score = start + (change * i) + np.random.normal(0, noise)
        score = max(5, min(100, score)) #keep between 5 and 100
        rows.append({'year': year, 'flavor': flavor, 'category': category, 'popularity_index': round(score, 1)})

df = pd.DataFrame(rows)
print(f'Dataset Shape: {df.shape}')
df.head(10)

Dataset Shape: (154, 4)


,year,flavor,category,popularity_index
0,2014,Vanilla,Traditional,89.0
1,2015,Vanilla,Traditional,85.9
2,2016,Vanilla,Traditional,85.7
3,2017,Vanilla,Traditional,85.6
4,2018,Vanilla,Traditional,80.3
5,2019,Vanilla,Traditional,78.5
6,2020,Vanilla,Traditional,80.4
7,2021,Vanilla,Traditional,76.9
8,2022,Vanilla,Traditional,72.7
9,2023,Vanilla,Traditional,72.9


### 3. Quick EDA

In [5]:
print('Categories:')
print(df['category'].value_counts())
print()
print('Popularity index stats:')
print(df['popularity_index'].describe().round(2))

Categories:
category
Traditional      44
Spice/Nutty      44
Floral           33
Citrus/Global    33
Name: count, dtype: int64

Popularity index stats:
count    154.00
mean      53.61
std       18.17
min        8.80
25%       41.72
50%       56.65
75%       67.00
max       89.00
Name: popularity_index, dtype: float64


In [6]:
# Average score per flavor in 2024 vs 2014
start_scores = df[df['year'] == 2014][['flavor', 'popularity_index']].rename(columns={'popularity_index': 'score_2014'})
end_scores = df[df['year'] == 2024][['flavor', 'popularity_index']].rename(columns={'popularity_index': 'score_2024'})

comparison = start_scores.merge(end_scores, on='flavor')
comparison['change'] = (comparison['score_2024'] - comparison['score_2014']).round(1)
comparison = comparison.sort_values('change', ascending=False)

print('Biggest movers 2014 - 2024')
comparison

Biggest movers 2014 - 2024


,flavor,score_2014,score_2024,change
8,Ube,8.8,75.0,66.2
7,Yuzu,11.4,70.7,59.3
11,Matcha,23.2,77.1,53.9
10,Cardamom,17.1,69.2,52.1
6,Elderflower,17.9,60.2,42.3
9,Passion Fruit,21.5,61.9,40.4
4,Lavender,27.0,67.1,40.1
12,Pistachio,33.1,70.1,37.0
5,Rose,30.3,61.4,31.1
13,Cinnamon,60.4,55.5,-4.9


### 4. Calculate Year-over-Year Growth (Momentum)

In [7]:
df = df.sort_values(['flavor', 'year'])
df['yoy_change'] = df.groupby('flavor')['popularity_index'].diff()
df['yoy_pct']    = df.groupby('flavor')['popularity_index'].pct_change().mul(100).round(2)

print('YoY columns added. Sample:')
df[df['flavor'] == 'Yuzu'].head()

YoY columns added. Sample:


,year,flavor,category,popularity_index,yoy_change,yoy_pct
77,2014,Yuzu,Citrus/Global,11.4,NaN,NaN
78,2015,Yuzu,Citrus/Global,18.0,6.6,57.89
79,2016,Yuzu,Citrus/Global,19.6,1.6,8.89
80,2017,Yuzu,Citrus/Global,29.0,9.4,47.96
81,2018,Yuzu,Citrus/Global,35.9,6.9,23.79


### 5. Interactive Time-Series - Filet by Category

In [8]:
fig = px.line(
    df, x='year', y='popularity_index',
    color='flavor', facet_col='category',
    facet_col_wrap=2,
    markers=True,
    title='Flavor Popularity index by category (2014-2024)',
    labels={'popularity_index': 'Popularity Index', 'year': 'Year'},
    hover_data={'yoy_pct': True, 'category': False},
    template='plotly_white',
    height=700,
)
fig.update_traces(line=dict(width=2.5))
fig.update_layout(
    font_family= 'Georgia',
    title_font_size = 16,
    legend_title ='Flavor',
    hoverlabel=dict(bgcolor='white', font_size=12),
)

fig.show()

### 6. All Flavors - Single Interactive Chart

In [9]:
fig2 = px.line(
    df, x='year', y='popularity_index',
    color='flavor', line_group='flavor',
    markers=True,
    title='All Flavors - Popularity Trends (hover for YoY % Change)',
    labels={'popularity_index': 'Popularity Index', 'year': 'Year'},
    hover_data={'yoy_pct': ':.1f', 'category': False},
    template='plotly_white',
    height=550,
)

fig2.update_layout(
    font_family= 'Georgia',
    title_font_size = 15,
    hovermode= 'x unified',
    legend=dict(orientation='v', x=1.01),
)

fig2.show()

### 7. Momentum Dashboard
This section highlights the top 5 **growth flavors** and the 3 **fading flavors** based on total change from 2014 to 2024.

In [10]:
top_growth = comparison.head(5)
fading = comparison.tail(3)

print('TOP 5 GROWTH FLAVORS:')
print(top_growth[['flavor', 'score_2014', 'score_2024', 'change']].to_string(index=False))
print()
print('TOP 3 FADING FLAVORS:')
print(fading[['flavor', 'score_2014', 'score_2024', 'change']].to_string(index=False))

TOP 5 GROWTH FLAVORS:
     flavor  score_2014  score_2024  change
        Ube         8.8        75.0    66.2
       Yuzu        11.4        70.7    59.3
     Matcha        23.2        77.1    53.9
   Cardamom        17.1        69.2    52.1
Elderflower        17.9        60.2    42.3

TOP 3 FADING FLAVORS:
    flavor  score_2014  score_2024  change
 Chocolate        83.8        72.4   -11.4
Strawberry        75.1        60.0   -15.1
   Vanilla        89.0        69.1   -19.9


In [14]:
# Momentum bar Chart
momentum_df = comparison.copy()
momentum_df['color'] = momentum_df['change'].apply(lambda x: 'Growing' if x>0 else 'Fading')
momentum_df = momentum_df.sort_values('change')

fig3 = px.bar(
    momentum_df, x='change', y='flavor',
    color='color',
    color_discrete_map={'Growing': '#2a9d8f', 'Fading': '#e76f51'},
    orientation='h',
    title='Momentum Dashboard - Total Popularity Change 2014 to 2024',
    labels={'change': ' Change in popularity index', 'flavor': 'Flavor'},
    template='plotly_white',
    height=500,
    hover_data={'score_2014': True, 'score_2024': True},
)

fig3.update_layout(
    font_family= 'Georgia',
    title_font_size = 15,
    legend_title='Trend',
    hoverlabel=dict(bgcolor='white')
)

fig3.add_vline(x=0, line_dash='dash', line_color='grey')
fig3.show()

In [15]:
#Categoriy-level average trend
cat_df = df.groupby(['year','category'])['popularity_index'].mean().reset_index()

fig4 = px.area(
    cat_df, x='year', y='popularity_index',
    color='category',
    title='Average Popularity by Flavor Category Over Time',
    labels={'popularity_index': 'Avg Popuarity Index', 'year': 'Year'},
    template='plotly_white',
    height=450,
    color_discrete_map={
        'Traditional': '#e9c46a',
        'Floral': '#a8dadc',
        'Citrus/Golbal': '#e76f51',
        'Spice/Nutty': '#2a9d8f',
    }
)

fig4.update_layout(font_family='Georgia', title_font_size=15)
fig4.show()

### 8. Export CSV

In [16]:
df.to_csv('velvet_yeast_falvour_trends.csv', index=False)
print(f'Exported {len(df)} rows to velvet_yeast_flavour_trends.csv')
df.head()

Exported 154 rows to velvet_yeast_flavour_trends.csv


,year,flavor,category,popularity_index,yoy_change,yoy_pct
110,2014,Cardamom,Spice/Nutty,17.1,NaN,NaN
111,2015,Cardamom,Spice/Nutty,24.8,7.7,45.03
112,2016,Cardamom,Spice/Nutty,29.7,4.9,19.76
113,2017,Cardamom,Spice/Nutty,38.1,8.4,28.28
114,2018,Cardamom,Spice/Nutty,38.9,0.8,2.10


### 9. Export Interactive HTML

In [17]:
import plotly.io as pio

# Combine all charts into one HTML file
with open('velvet_yeast_dashboard.html', 'w') as f:
    f.write('<html><head><title>Velvet & Yeast — Flavor Trend Dashboard</title>')
    f.write('<style>body{font-family:Georgia;background:#fafafa;padding:30px;} h1{color:#3d2b1f;} p{color:#555;max-width:800px;}</style></head><body>')
    f.write('<h1>Velvet & Yeast Patisserie — Flavor Trend Dashboard</h1>')
    f.write('<p>Interactive dashboard showing consumer flavor preference trends from 2014 to 2024. Hover over any data point to see year-over-year percentage changes.</p>')
    f.write(pio.to_html(fig2, full_html=False, include_plotlyjs='cdn'))
    f.write(pio.to_html(fig3, full_html=False, include_plotlyjs=False))
    f.write(pio.to_html(fig,  full_html=False, include_plotlyjs=False))
    f.write(pio.to_html(fig4, full_html=False, include_plotlyjs=False))
    f.write('</body></html>')

print('HTML dashboard exported: velvet_yeast_dashboard.html')

HTML dashboard exported: velvet_yeast_dashboard.html
